# Interactive Dashboard — Plotly
**ENGR 010 Group Project — Power Systems Analysis and Monitoring**

This notebook builds an interactive dashboard using `plotly`. Every chart supports zooming, panning, hover tooltips, and clicking legend items to toggle stations on/off — all without any extra widgets. Analysis logic is imported from `analysis.py`.

In [1]:
import sys
!{sys.executable} -m pip install plotly nbformat -q


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: pip3.12 install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import analysis

df = analysis.load_data('power_system_data.csv')
df['month_name'] = df['timestamp'].dt.strftime('%b')
df['hour']       = df['timestamp'].dt.hour
df['month']      = df['timestamp'].dt.month

COLORS = {'SUB_001': '#4e8bc4', 'SUB_002': '#e07b5e', 'SUB_003': '#5bab7a'}
STATIONS = sorted(df['station_id'].unique().tolist())

print('Data loaded. Plotly dashboard ready.')

Loaded 13,035 records from 'power_system_data.csv'.
Data loaded. Plotly dashboard ready.


---
## Plot 1 — Interactive Time Series: All Parameters

Use the dropdown menu built into the chart to switch between parameters. Click station names in the legend to toggle them. Zoom by clicking and dragging.

In [3]:
params = {
    'real_power_mw':       'Real Power (MW)',
    'reactive_power_mvar': 'Reactive Power (MVAR)',
    'voltage_pu':          'Voltage (pu)',
    'current_pu':          'Current (pu)',
    'power_factor':        'Power Factor',
}

fig = go.Figure()

for p_idx, (param, label) in enumerate(params.items()):
    for sid in STATIONS:
        s = df[df['station_id'] == sid].sort_values('timestamp')
        fig.add_trace(go.Scatter(
            x=s['timestamp'],
            y=s[param],
            name=sid,
            line=dict(color=COLORS[sid], width=1),
            visible=(p_idx == 0),
            legendgroup=sid,
            showlegend=(p_idx == 0),
            hovertemplate=f'<b>{sid}</b><br>%{{x|%b %d %H:%M}}<br>{label}: %{{y:.3f}}<extra></extra>',
        ))

n_stations = len(STATIONS)

buttons = []
for p_idx, (param, label) in enumerate(params.items()):
    visible = [False] * (len(params) * n_stations)
    for i in range(n_stations):
        visible[p_idx * n_stations + i] = True
    buttons.append(dict(
        label=label,
        method='update',
        args=[
            {'visible': visible},
            {'yaxis': {'title': label}},
        ],
    ))

first_label = list(params.values())[0]
fig.update_layout(
    title='Power System Time Series — All Substations',
    xaxis_title='Time',
    yaxis_title=first_label,
    height=450,
    hovermode='x unified',
    updatemenus=[dict(
        type='dropdown',
        x=0.0, y=1.18,
        showactive=True,
        buttons=buttons,
        bgcolor='white',
        bordercolor='lightgray',
    )],
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    template='plotly_white',
)

fig.show()

---
## Plot 2 — Four-Panel Monitoring Dashboard

A single figure with four synchronized subplots — voltage, real power, power factor, and reactive power — all sharing the same x-axis. Zoom on any panel and the rest follow.

In [4]:
panel_params = [
    ('voltage_pu',          'Voltage (pu)'),
    ('real_power_mw',       'Real Power (MW)'),
    ('power_factor',        'Power Factor'),
    ('reactive_power_mvar', 'Reactive Power (MVAR)'),
]

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=[label for _, label in panel_params],
    vertical_spacing=0.04,
)

for row, (param, label) in enumerate(panel_params, start=1):
    for sid in STATIONS:
        s = df[df['station_id'] == sid].sort_values('timestamp')
        fig.add_trace(
            go.Scatter(
                x=s['timestamp'],
                y=s[param],
                name=sid,
                line=dict(color=COLORS[sid], width=0.8),
                legendgroup=sid,
                showlegend=(row == 1),
                hovertemplate=f'<b>{sid}</b><br>%{{y:.3f}}<extra></extra>',
            ),
            row=row, col=1,
        )

    if param == 'voltage_pu':
        for limit, name in [(0.95, 'Lower Limit'), (1.05, 'Upper Limit')]:
            fig.add_hline(y=limit, line_dash='dash', line_color='orange',
                          annotation_text=name, annotation_position='top right',
                          row=row, col=1)
    if param == 'power_factor':
        fig.add_hline(y=0.90, line_dash='dash', line_color='orange',
                      annotation_text='Min PF = 0.90', annotation_position='top right',
                      row=row, col=1)

fig.update_layout(
    height=900,
    title_text='Four-Panel Grid Monitoring Dashboard',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.1, xanchor='right', x=1),
    margin=dict(t=100),
)

fig.show()

---
## Plot 3 — Load Heatmap

Average real power by hour of day and month. Hover over any cell for the exact value.

In [5]:
pivot = (
    df.groupby(['month', 'hour'])['real_power_mw']
    .mean()
    .unstack()
)

month_names = ['Jan','Feb','Mar','Apr','May','Jun']
y_labels    = [month_names[m - 1] for m in pivot.index]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=list(pivot.columns),
    y=y_labels,
    colorscale='YlOrRd',
    colorbar=dict(title='Avg MW'),
    hovertemplate='Hour: %{x}<br>Month: %{y}<br>Avg Power: %{z:.1f} MW<extra></extra>',
))

fig.update_layout(
    title='Load Heatmap — Average Real Power by Hour and Month',
    xaxis_title='Hour of Day',
    yaxis_title='Month',
    height=350,
    template='plotly_white',
)

fig.show()

---
## Plot 4 — Power Triangle (Interactive)

One power triangle per station using its mean values. Hover over each arrow to see the exact magnitude.

In [6]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=STATIONS,
)

for col, sid in enumerate(STATIONS, start=1):
    s  = df[df['station_id'] == sid]
    P  = s['real_power_mw'].mean()
    Q  = s['reactive_power_mvar'].mean()
    S  = np.sqrt(P**2 + Q**2)
    PF = P / S

    color = COLORS[sid]

    fig.add_trace(go.Scatter(
        x=[0, P], y=[0, Q],
        mode='lines+markers',
        line=dict(color=color, width=3),
        name=f'{sid} S',
        hovertemplate=f'Apparent Power S = {S:.1f} MVA<extra></extra>',
        showlegend=False,
    ), row=1, col=col)

    fig.add_trace(go.Scatter(
        x=[0, P], y=[0, 0],
        mode='lines',
        line=dict(color='seagreen', width=3),
        name=f'{sid} P',
        hovertemplate=f'Real Power P = {P:.1f} MW<extra></extra>',
        showlegend=False,
    ), row=1, col=col)

    fig.add_trace(go.Scatter(
        x=[P, P], y=[0, Q],
        mode='lines',
        line=dict(color='tomato', width=3),
        name=f'{sid} Q',
        hovertemplate=f'Reactive Power Q = {Q:.1f} MVAR<extra></extra>',
        showlegend=False,
    ), row=1, col=col)

    fig.add_annotation(
        x=P / 2, y=Q + Q * 0.12,
        text=f'PF = {PF:.3f}<br>S = {S:.1f} MVA',
        showarrow=False,
        font=dict(size=10),
        row=1, col=col,
    )

fig.update_layout(
    title='Power Triangle — Mean Values per Substation',
    height=400,
    template='plotly_white',
)
fig.update_xaxes(title_text='Real Power (MW)')
fig.update_yaxes(title_text='Reactive Power (MVAR)')

fig.show()

---
## Plot 5 — Fault Detection Timeline

Voltage time series with detected fault events marked in red. Hover to see the exact timestamp, voltage, and fault type for any event.

In [7]:
faults = analysis.detect_faults(df)

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=STATIONS,
    vertical_spacing=0.06,
)

for row, sid in enumerate(STATIONS, start=1):
    s = df[df['station_id'] == sid].sort_values('timestamp')

    fig.add_trace(go.Scatter(
        x=s['timestamp'], y=s['voltage_pu'],
        mode='lines',
        name=sid,
        line=dict(color=COLORS[sid], width=0.8),
        legendgroup=sid,
        showlegend=True,
        hovertemplate='%{x|%b %d %H:%M}<br>Voltage: %{y:.4f} pu<extra></extra>',
    ), row=row, col=1)

    sf = faults[faults['station_id'] == sid]
    if not sf.empty:
        fig.add_trace(go.Scatter(
            x=sf['timestamp'], y=sf['voltage_pu'],
            mode='markers',
            name='Fault',
            marker=dict(color='red', size=6, symbol='x'),
            legendgroup='fault',
            showlegend=(row == 1),
            hovertemplate='<b>FAULT: %{customdata}</b><br>%{x|%b %d %H:%M}<br>Voltage: %{y:.4f} pu<extra></extra>',
            customdata=sf['fault_type'],
        ), row=row, col=1)

    for limit in [0.95, 1.05]:
        fig.add_hline(
            y=limit, line_dash='dash', line_color='orange', line_width=1,
            row=row, col=1,
        )

fig.update_layout(
    height=750,
    title_text='Voltage Time Series with Fault Detection',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.1, xanchor='right', x=1),
    margin=dict(t=100),
)
fig.update_yaxes(title_text='Voltage (pu)')

fig.show()

---
## Plot 6 — Grid Health Score

Composite health score per station. Hover for the exact score and status.

In [8]:
health_scores = analysis.calculate_grid_health_score(df)

stations = list(health_scores.keys())
scores   = list(health_scores.values())
statuses = ['GOOD' if s >= 90 else 'FAIR' if s >= 75 else 'POOR' for s in scores]
bar_colors = ['seagreen' if s >= 90 else 'darkorange' if s >= 75 else 'crimson'
              for s in scores]

fig = go.Figure(go.Bar(
    x=stations,
    y=scores,
    marker_color=bar_colors,
    text=[f'{s:.1f}' for s in scores],
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Score: %{y:.1f}<br>Status: %{customdata}<extra></extra>',
    customdata=statuses,
))

fig.add_hline(y=90, line_dash='dash', line_color='seagreen',
              annotation_text='Good (≥90)', annotation_position='top right')
fig.add_hline(y=75, line_dash='dash', line_color='darkorange',
              annotation_text='Fair (≥75)', annotation_position='top right')

fig.update_layout(
    title='Composite Grid Health Score by Substation',
    yaxis=dict(title='Health Score (0–100)', range=[0, 110]),
    height=400,
    template='plotly_white',
)

fig.show()